# 05. Közösségi Infrastruktúra és a 15 Perces Város

**Cél:** az ingatlanok intézményi és kereskedelmi ellátottságának (POI) mérése a „15 perces város" koncepció tükrében.

---

### 📖 A 15 perces város elmélete
Carlos Moreno (2016) koncepciója szerint az ideális városi szövetben a lakosok minden alapvető funkciót elérnek 15 perces aktív közlekedéssel. Az ilyen területeken az ingatlanok „walkability" (sétálhatósági) prémiummal rendelkeznek.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from ingatlan_tdk._utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
try:
    import osmnx as ox
    import geopandas as gpd
    from shapely.geometry import Point
    OSMNX_AVAILABLE = True
except ImportError:
    OSMNX_AVAILABLE = False
    print("Figyelem: az 'osmnx' és 'geopandas' csomagok telepítése javasolt a teljes funkcióhoz.")

df = load_szamitott_master()
display(HTML(render_area_header_html(df=df)))
df_pontos = df[(df['minta_garantalt_pontos'] == 1) & (df['listing_type'] == 'elado')].copy()
print(f"Elemzett minta (garantált pontos): {len(df_pontos)} db.")

### 1. POI Sűrűség és Ellátottsági Térképek

**📌 Kiszámítási módszertan:**
Az OpenStreetMap adatbázisból kinyert funkciók (boltok, rendelők, iskolák, parkok) térbeli sűrűségét vizsgáljuk; a sűrűsödési magok a 15 perces város lokális centrumait jelölik.

**🔍 Városgazdasági hatás:**
A magas intézményi sűrűség csökkenti a lakosok tranzakciós költségeit (idő, pénz), ami tőkésedik az árakban; a szolgáltatási sivatagokban fekvő ingatlanok strukturális értékhátrányt szenvednek.

In [ ]:
# Pufferelt POI adatbázis betöltése (kontextus; a POI-sávok a kanonikus adatban vannak)
import geopandas as gpd
area_id = os.environ.get('TDK_ACTIVE_AREA', 'kobanya')
cand_paths = [
    os.path.join('data', f'poi_{area_id}_buffered.geojson'),
    os.path.join('..', 'data', f'poi_{area_id}_buffered.geojson'),
    os.path.abspath(f'data/poi_{area_id}_buffered.geojson')
]
poi_cache_file = next((p for p in cand_paths if os.path.exists(p)), cand_paths[0])
if os.path.exists(poi_cache_file):
    poi_data = gpd.read_file(poi_cache_file)
    print(f"Pufferelt POI adatbázis: {len(poi_data)} db szolgáltatás (kánon: letöltési puffer 1125 m).")
else:
    poi_data = None
    print("POI geojson nem található — a POI-sávok a kanonikus master oszlopokból jönnek.")

# A KANONIKUS POI-sávok a masterben: poi_5p_count / poi_10p_count / poi_15p_count
# (VALÓDI hálózati elérhetőség, 0-375-750-1125 m, 1.25 m/s — a pipeline számította)


### 2. Pufferelt Hálózati Index

**📌 A határhatások (edge effect) korrekciója:**
Az adminisztratív határ közelében fekvő ingatlanoknál a határon belüli POI-számolás torzít — a lakó a szomszédos terület szolgáltatóit is eléri. Ezt hálózati pufferezéssel és szélesebb lekérdezéssel küszöböljük ki, reális képet adva a peremzónák ellátottságáról.

In [ ]:
# POI-sávok a kanonikus adatból (hálózati elérhetőség: 375/750/1125 m)
df_pontos = df_pontos.dropna(subset=['poi_5p_count', 'poi_10p_count', 'poi_15p_count'])

kpi_cards = [
    ("Átlagos POI 5p (≤375m)", f"{df_pontos['poi_5p_count'].mean():.1f} db", "Közvetlen környezet (hálózat)", "#1e3a8a"),
    ("Átlagos POI 10p (≤750m)", f"{df_pontos['poi_10p_count'].mean():.1f} db", "Napi szükségletek (hálózat)", "#059669"),
    ("Átlagos POI 15p (≤1125m)", f"{df_pontos['poi_15p_count'].mean():.1f} db", "15 perces város zóna (hálózat)", "#2563eb"),
    ("Maximum POI (15p)", f"{df_pontos['poi_15p_count'].max():.0f} db", "Legjobban ellátott pont", "#7c3aed"),
    ("Minimum POI (15p)", f"{df_pontos['poi_15p_count'].min():.0f} db", "Periféria / ipari zóna", "#dc2626")
]
display(HTML(kpi_grid_html(kpi_cards)))

df_map = df_pontos.dropna(subset=['nm_ar_huf', 'poi_15p_count', 'geokodolt_lat', 'geokodolt_lon'])
fig = px.scatter_map(
    df_map, lat='geokodolt_lat', lon='geokodolt_lon', color='poi_15p_count',
    size='nm_ar_huf', hover_name='cim_teljes', map_style='carto-positron',
    color_continuous_scale='Viridis',
    title='"15 perces város" - Hálózaton elérhető szolgáltatások száma 1125 méteren belül'
)
fig.update_layout(height=500, margin={"r":0,"t":40,"l":0,"b":0})
fig.show()


### 3. A Szolgáltatási Sűrűség Hedonikus Árprémiuma

**📌 Mit mutat a regressziós modell?**
A hedonikus alapmodellt (méret, szobaszám, panel, lift, állapot, erkély, emelet, metró- és vasútállomás-távolság) három POI-sávval bővítjük (5/10/15 perces HÁLÓZATI elérhetőség, 375/750/1125 m). A POI-sávok kumulatívak, ezért a kollinearitás elkerülésére külön-külön és együtt is becsüljük őket; az összehasonlítást F-teszt, a standard hibákat HC1 robusztus becslés adja. A modell a garantáltan pontos eladó almintán fut (a térbeli változók csak ott definiáltak — ez strukturális, nem adathiány).


In [ ]:
import statsmodels.api as sm

# Egységesített kontrollváltozók
df_pontos['emelet_szam'] = df_pontos['emelet_szam'].fillna(df_pontos['emelet_szam'].median())
df_pontos['allapot_kod'] = df_pontos['allapot_kod'].fillna(df_pontos['allapot_kod'].median())

base_features = [
    'korrigalt_alapterulet_nm', 'szobaszam_osszes', 'is_panel',
    'has_lift', 'allapot_kod', 'has_erkely', 'emelet_szam',
    'tavolsag_metro_halozati_m', 'tavolsag_vasut_halozati_m'
]
poi_vars = ['poi_5p_count', 'poi_10p_count', 'poi_15p_count']

df_reg = df_pontos.dropna(subset=['log_nm_ar'] + base_features + poi_vars).copy()
y = df_reg['log_nm_ar']

X_base = sm.add_constant(df_reg[base_features])
model_base = sm.OLS(y, X_base).fit(cov_type='HC1')

# Kiterjesztett modellek: POI-sávonként (külön, a kumulatív kollinearitás elkerülésére)
models_ext = {}
for pv in poi_vars:
    models_ext[pv] = sm.OLS(y, sm.add_constant(df_reg[base_features + [pv]])).fit(cov_type='HC1')

# F-teszt: alapmodell vs. kiterjesztett (mindhárom POI-sáv együtt)
X_all = sm.add_constant(df_reg[base_features + poi_vars])
# Az F-teszthez közönséges (nem robusztus) kovarianciájú illesztések kellenek
# (a robust kovarianciával a compare_f_test érvénytelen figyelmeztetést ad).
model_all_plain = sm.OLS(y, X_all).fit()
model_base_plain = sm.OLS(y, X_base).fit()
f_stat, f_p, _ = model_all_plain.compare_f_test(model_base_plain)

var_labels = {
    'const': 'Tengelymetszet (Konstans)',
    'korrigalt_alapterulet_nm': 'Korrigált alapterület (m²)',
    'szobaszam_osszes': 'Szobaszám',
    'is_panel': 'Panelszerkezet dummy',
    'has_lift': 'Lift dummy',
    'allapot_kod': 'Műszaki állapot index',
    'has_erkely': 'Erkély dummy',
    'emelet_szam': 'Emelet szintszám',
    'tavolsag_metro_halozati_m': 'Metróállomás hálózati táv. (m)',
    'tavolsag_vasut_halozati_m': 'Vasútállomás hálózati táv. (m)',
    'poi_5p_count': 'POI 5p (≤375m, hálózat)',
    'poi_10p_count': 'POI 10p (≤750m, hálózat)',
    'poi_15p_count': 'POI 15p (≤1125m, hálózat)'
}

cmp_rows = []
all_vars = ['const'] + base_features + poi_vars
for v in all_vars:
    row = {'Változó': var_labels.get(v, v)}
    row['Alapmodell'] = (f"{model_base.params[v]:.5f} (p={model_base.pvalues[v]:.3f})"
                         if v in model_base.params else '-')
    row['+ POI 5p'] = (f"{models_ext['poi_5p_count'].params[v]:.5f} (p={models_ext['poi_5p_count'].pvalues[v]:.3f})"
                       if v in models_ext['poi_5p_count'].params else '-')
    row['+ POI 15p'] = (f"{models_ext['poi_15p_count'].params[v]:.5f} (p={models_ext['poi_15p_count'].pvalues[v]:.3f})"
                        if v in models_ext['poi_15p_count'].params else '-')
    cmp_rows.append(row)

stat_rows = [
    {'Változó': 'R²', 'Alapmodell': f"{model_base.rsquared:.3f}",
     '+ POI 5p': f"{models_ext['poi_5p_count'].rsquared:.3f}",
     '+ POI 15p': f"{models_ext['poi_15p_count'].rsquared:.3f}"},
    {'Változó': 'Adj. R²', 'Alapmodell': f"{model_base.rsquared_adj:.3f}",
     '+ POI 5p': f"{models_ext['poi_5p_count'].rsquared_adj:.3f}",
     '+ POI 15p': f"{models_ext['poi_15p_count'].rsquared_adj:.3f}"},
    {'Változó': 'AIC', 'Alapmodell': f"{model_base.aic:.1f}",
     '+ POI 5p': f"{models_ext['poi_5p_count'].aic:.1f}",
     '+ POI 15p': f"{models_ext['poi_15p_count'].aic:.1f}"},
    {'Változó': 'N', 'Alapmodell': f"{int(model_base.nobs)}",
     '+ POI 5p': f"{int(models_ext['poi_5p_count'].nobs)}",
     '+ POI 15p': f"{int(models_ext['poi_15p_count'].nobs)}"},
]
display(HTML(pd.DataFrame(cmp_rows).to_html(index=False)))
display(HTML(pd.DataFrame(stat_rows).to_html(index=False)))
print(f"F-teszt (alap vs. mindhárom POI-sáv): F={f_stat:.2f}, p={f_p:.4f}")

# POI-együtthatók szignifikanciával (HC1 SE-k)
print("\nPOI-együtthatók (HC1 robusztus SE):")
for pv in poi_vars:
    m = models_ext[pv]
    b, pv_ = m.params[pv], m.pvalues[pv]
    print(f"  {var_labels[pv]}: b={b:.5f} (p={pv_:.3f}) -> 1 POI hatása: {(np.exp(b)-1)*100:+.2f}%")
